# Lab: Principal Component Analysis and Data Governance

## Overview
This lab explores Principal Component Analysis (PCA) as a tool for understanding high-dimensional data. More importantly, we examine how data reduction choices affect what we *can* and *cannot* see in datasets, with implications for governance and fairness.

**Learning Goals:**
- Understand PCA as a dimensionality reduction technique
- Calculate and interpret principal components by hand
- Visualize high-dimensional data in 2D
- Reflect on how data choices shape governance decisions

## Part 1: Why Data Governance Matters

Governments and platforms collect massive amounts of data about people: demographics, location, behavior, health, voting patterns, and more. When policymakers need to understand communities, they often use data reduction techniques like PCA to identify the most important patterns.

**Key Questions:**
- Which features matter most for the decision being made?
- What gets lost when we project high-dimensional data into fewer dimensions?
- Who is represented (or not) in the data collection process?
- How might different PCA choices lead to different policy recommendations?

This lab builds the mathematical intuition; later homeworks explore real governance scenarios.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Set random seed for reproducibility
np.random.seed(42)

## Part 2: A Simple Example – Neighborhood Data

Imagine a city agency collects data on neighborhoods across four features:
- Average household income (in thousands)
- Public transit ridership (percentage of commutes)
- Broadband access (percentage of households)
- Healthcare clinic visits (per capita, annual)

Let's create a small dataset:

In [ ]:
# Create neighborhood data
neighborhoods = ['Downtown', 'Midtown', 'Suburbs', 'Hills', 'Valley']

data = np.array([
    [65, 70, 95, 12],   # Downtown: high income, high transit, good broadband, many clinic visits
    [45, 40, 85, 18],   # Midtown: medium income, medium transit, good broadband, more clinic visits
    [35, 15, 60, 24],   # Suburbs: lower income, low transit, moderate broadband, more clinic visits
    [85, 10, 92, 8],    # Hills: high income, low transit, good broadband, few clinic visits
    [30, 50, 50, 28]    # Valley: low income, moderate transit, moderate broadband, many clinic visits
])

# Feature names
features = ['Income (K)', 'Transit (%)', 'Broadband (%)', 'Clinic Visits']

# Create DataFrame for easy viewing
df = pd.DataFrame(data, columns=features, index=neighborhoods)
print("Neighborhood Data:")
print(df)
print()

## Part 3: Centering and Standardizing the Data

PCA works on centered, standardized data. We subtract the mean and divide by standard deviation so each feature has comparable scale.

In [ ]:
# Center the data
data_centered = data - np.mean(data, axis=0)

print("Centered data (mean subtracted):")
print(data_centered)
print()

# Standardize (divide by standard deviation)
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

print("Standardized data:")
print(data_scaled)
print()
print(f"Mean of scaled data: {np.mean(data_scaled, axis=0)}")
print(f"Std of scaled data: {np.std(data_scaled, axis=0)}")

## Part 4: Computing the Covariance Matrix

The covariance matrix shows how features relate to each other. Large positive values mean features tend to increase together; negative means they move opposite.

In [ ]:
# Compute covariance matrix
cov_matrix = np.cov(data_scaled.T)

print("Covariance matrix:")
print(cov_matrix)
print()

# Visualize as heatmap
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cov_matrix, cmap='coolwarm', vmin=-1, vmax=1)

# Labels
ax.set_xticks(range(len(features)))
ax.set_yticks(range(len(features)))
ax.set_xticklabels(features, rotation=45, ha='right')
ax.set_yticklabels(features)

# Add text annotations
for i in range(len(features)):
    for j in range(len(features)):
        text = ax.text(j, i, f'{cov_matrix[i, j]:.2f}',
                       ha="center", va="center", color="black", fontsize=11)

ax.set_title('Covariance Matrix\n(How features relate to each other)', fontsize=12)
plt.colorbar(im, ax=ax, label='Covariance')
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Positive values: features increase together")
print("- Negative values: features move in opposite directions")
print("- Larger magnitude: stronger relationship")

## Part 5: Principal Components via Eigendecomposition

The principal components are the eigenvectors of the covariance matrix. The eigenvalues tell us how much variance each component captures.

In [ ]:
# Compute eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# Sort by decreasing eigenvalue
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print("Eigenvalues (variance explained by each component):")
for i, ev in enumerate(eigenvalues):
    variance_explained = 100 * ev / np.sum(eigenvalues)
    print(f"  PC{i+1}: {ev:.4f} ({variance_explained:.2f}% of variance)")

print("\nPrincipal Component directions (eigenvectors):")
pc_df = pd.DataFrame(
    eigenvectors,
    columns=[f'PC{i+1}' for i in range(len(features))],
    index=features
)
print(pc_df)

# Cumulative variance explained
cumsum_var = np.cumsum(eigenvalues) / np.sum(eigenvalues)
print(f"\nCumulative variance explained:")
for i, cum_var in enumerate(cumsum_var):
    print(f"  First {i+1} component(s): {100*cum_var:.2f}%")

## Part 6: Projecting Data onto Principal Components

Now we project the original data onto the first two principal components to see neighborhoods in 2D.

In [ ]:
# Project data onto first 2 PCs
pc_scores = data_scaled @ eigenvectors[:, :2]

print("Scores on first two principal components:")
scores_df = pd.DataFrame(
    pc_scores,
    columns=['PC1', 'PC2'],
    index=neighborhoods
)
print(scores_df)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
colors = ['red', 'blue', 'green', 'orange', 'purple']

for i, neighborhood in enumerate(neighborhoods):
    ax.scatter(pc_scores[i, 0], pc_scores[i, 1], s=200, c=colors[i], alpha=0.7, label=neighborhood)
    ax.annotate(neighborhood, (pc_scores[i, 0], pc_scores[i, 1]), 
                xytext=(5, 5), textcoords='offset points', fontsize=10)

ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax.axvline(x=0, color='k', linestyle='--', alpha=0.3)

ax.set_xlabel(f'PC1 ({100*eigenvalues[0]/np.sum(eigenvalues):.1f}% variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({100*eigenvalues[1]/np.sum(eigenvalues):.1f}% variance)', fontsize=12)
ax.set_title('Neighborhoods in 2D PCA Space', fontsize=14)
ax.grid(True, alpha=0.3)
ax.legend(loc='best')
plt.tight_layout()
plt.show()

print("\nInterpretation: Each neighborhood's position reflects its similarity to others.")
print("Nearby neighborhoods have similar patterns across income, transit, broadband, and health.")

## Part 7: Understanding Principal Component Directions

What does each PC represent? We look at the loadings (contributions of original features).

In [ ]:
# Loadings = eigenvectors scaled by sqrt(eigenvalue)
loadings = eigenvectors[:, :2] * np.sqrt(eigenvalues[:2])

print("Loadings (contribution of each feature to each PC):")
loadings_df = pd.DataFrame(
    loadings,
    columns=['PC1', 'PC2'],
    index=features
)
print(loadings_df)

# Biplot
fig, ax = plt.subplots(figsize=(10, 8))

# Plot neighborhoods
for i, neighborhood in enumerate(neighborhoods):
    ax.scatter(pc_scores[i, 0], pc_scores[i, 1], s=200, c=colors[i], alpha=0.5)
    ax.annotate(neighborhood, (pc_scores[i, 0], pc_scores[i, 1]), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)

# Plot feature arrows
arrow_scale = 3
for i, feature in enumerate(features):
    ax.arrow(0, 0, loadings[i, 0]*arrow_scale, loadings[i, 1]*arrow_scale,
             head_width=0.1, head_length=0.1, fc='black', ec='black', alpha=0.7, linewidth=2)
    ax.text(loadings[i, 0]*arrow_scale*1.15, loadings[i, 1]*arrow_scale*1.15, feature,
            fontsize=11, fontweight='bold', ha='center')

ax.axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax.axvline(x=0, color='k', linestyle='--', alpha=0.3)

ax.set_xlabel(f'PC1 ({100*eigenvalues[0]/np.sum(eigenvalues):.1f}% variance)', fontsize=12)
ax.set_ylabel(f'PC2 ({100*eigenvalues[1]/np.sum(eigenvalues):.1f}% variance)', fontsize=12)
ax.set_title('PCA Biplot: Neighborhoods and Feature Directions', fontsize=14)
ax.grid(True, alpha=0.3)
ax.set_xlim(-3, 3)
ax.set_ylim(-2.5, 2.5)
plt.tight_layout()
plt.show()

print("\nBiplot Interpretation:")
print(f"PC1: Features with large loadings: {features[np.argmax(np.abs(loadings[:, 0]))]}, {features[np.argmax(np.abs(loadings[:, 1]))]}, ...")
print("     This captures the main variation in the data.")
print(f"\nPC2: The second-most important variation, capturing patterns not explained by PC1.")

## Part 8: Student Exercise 1 – Interpreting PCA Results

**Question:** Based on the PCA results above:

1. Which neighborhoods are most similar in the 2D space?
2. What does PC1 primarily measure (look at the loadings)?
3. If a policymaker wants to target neighborhoods for a transit expansion, which neighborhoods would PC1 suggest?

**Your Analysis:** Write your answers in the cell below.

In [ ]:
# Compute distances between neighborhoods in PC space
from scipy.spatial.distance import pdist, squareform

distances = squareform(pdist(pc_scores, metric='euclidean'))
dist_df = pd.DataFrame(distances, index=neighborhoods, columns=neighborhoods)

print("Euclidean distances between neighborhoods in PC space:")
print(dist_df.round(2))
print()

# Find most similar pairs
min_dist = np.inf
pair = (None, None)
for i in range(len(neighborhoods)):
    for j in range(i+1, len(neighborhoods)):
        if distances[i, j] < min_dist:
            min_dist = distances[i, j]
            pair = (neighborhoods[i], neighborhoods[j])

print(f"Most similar neighborhoods: {pair[0]} and {pair[1]} (distance: {min_dist:.2f})")
print()

# PC1 interpretation
print("PC1 loadings (interpretation):")
pc1_sorted = np.argsort(loadings[:, 0])[::-1]
for idx in pc1_sorted:
    print(f"  {features[idx]:20s}: {loadings[idx, 0]:+.3f}")

## Part 9: Follow-Up Exploration – What Gets Lost?

When we project 4D data onto 2D, we lose information. How much, and what kind?

In [ ]:
# Variance explained by different numbers of components
cumsum_variance = np.cumsum(eigenvalues) / np.sum(eigenvalues)

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(range(1, len(cumsum_variance)+1), 100*cumsum_variance, alpha=0.7, color='steelblue')
ax.plot(range(1, len(cumsum_variance)+1), 100*cumsum_variance, 'ro-', linewidth=2, markersize=8)

ax.set_xlabel('Number of Components', fontsize=12)
ax.set_ylabel('Cumulative Variance Explained (%)', fontsize=12)
ax.set_title('How Much Information is Retained at Each Dimension?', fontsize=14)
ax.set_xticks(range(1, len(cumsum_variance)+1))
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 105)

for i, var in enumerate(cumsum_variance):
    ax.text(i+1, 100*var + 2, f'{100*var:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print("Information Loss Analysis:")
print("-" * 50)
print(f"Using 1 PC: {100*cumsum_variance[0]:.1f}% of variance; {100*(1-cumsum_variance[0]):.1f}% lost")
print(f"Using 2 PCs: {100*cumsum_variance[1]:.1f}% of variance; {100*(1-cumsum_variance[1]):.1f}% lost")
print(f"Using 3 PCs: {100*cumsum_variance[2]:.1f}% of variance; {100*(1-cumsum_variance[2]):.1f}% lost")
print(f"Using 4 PCs: {100*cumsum_variance[3]:.1f}% of variance; {100*(1-cumsum_variance[3]):.1f}% lost")

## Part 10: Policy Reflection – Data Governance

**Reflection Exercise:**

1. **What happens when we choose which features to measure?**
   - If a city only tracks *income* and *clinic visits*, what pattern might be missed?
   - Write your reflection here.

2. **How could PCA results influence policy differently depending on who performs the analysis?**
   - What if a neighborhood advocacy group wanted to emphasize transit access vs. an income-focused lens?
   - Write your reflection here.

3. **What are the ethical implications of dimensionality reduction in governance?**
   - Who decides which dimensions are "most important"?
   - Write your reflection here.

## Part 11: Generate Your Lab Report

In [ ]:
from datetime import datetime

report = f"""
================================================================================
LAB: PRINCIPAL COMPONENT ANALYSIS AND DATA GOVERNANCE
================================================================================
Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

DATASET: {len(neighborhoods)} neighborhoods, {len(features)} features

VARIANCE EXPLAINED BY EACH COMPONENT:
"""

for i, ev in enumerate(eigenvalues):
    variance_explained = 100 * ev / np.sum(eigenvalues)
    report += f"  PC{i+1}: {variance_explained:.2f}%\n"

report += f"""
KEY FINDINGS:
- First 2 components explain {100*cumsum_variance[1]:.1f}% of variance
- Most similar neighborhoods: [Your answer here]
- PC1 primarily measures: [Your answer here]

GOVERNANCE IMPLICATIONS:
- Dimensionality reduction simplifies but also abstracts
- Feature selection determines what is visible in the data
- PCA results depend on measurement choices and data collection practices

STUDENT REFLECTIONS:
[Your written reflections would appear here]

================================================================================
"""

print(report)

# Optionally save
# with open('lab-pca-report.txt', 'w') as f:
#     f.write(report)

## Summary

In this lab, you:
1. ✓ Learned how PCA identifies the most important patterns in data
2. ✓ Calculated principal components from a covariance matrix
3. ✓ Projected high-dimensional data into 2D for visualization
4. ✓ Analyzed what information is lost in dimensionality reduction
5. ✓ Reflected on how PCA choices affect governance decisions

**Next Steps:**
- Homework: Apply PCA to real demographic or sports data
- Advanced topic: t-SNE and non-linear dimensionality reduction
- Case study: How social media platforms use dimensionality reduction